# PestCLEF 2026: извлечение графа знаний только графовыми методами

Нейросетей и обучаемых классификаторов нет:
1. сущности ищутся и связываются по alias-рёбрам фонового графа знаний, подграфов GeoNames и NCBI Taxonomy и по иерархии мест;
2. отношения оцениваются Personalized PageRank на графе документа, объединённом с фоновым KG;
3. n-арные события — это максимальные клики графа отношений.

Код: https://github.com/neuraCollab/uni/tree/claude/laughing-hamilton-0c4nn3/competitions/pest-clef-2026

**Перед запуском:** подключите данные соревнования (Add Input → `pest-clef-2026`) и в Settings справа включите **Internet → On**. Нужен для клонирования кода и скачивания EPOP, GeoNames и NCBI. GPU не нужен, достаточно CPU; весь ноутбук выполняется примерно за 2 минуты.

In [ ]:
BRANCH = "claude/laughing-hamilton-0c4nn3"
!git clone -q --depth 1 --filter=blob:none --sparse -b $BRANCH https://github.com/neuraCollab/uni.git /kaggle/working/uni
!cd /kaggle/working/uni && git sparse-checkout set competitions/pest-clef-2026
%cd /kaggle/working/uni/competitions/pest-clef-2026
!pip install -q -r requirements.txt
# JSON соревнования (Add Input -> pest-clef-2026): dev для метрики Kaggle, test для списка doc_id
import glob, os
os.makedirs('data/kaggle', exist_ok=True)
for f in glob.glob('/kaggle/input/**/PestCLEF-2026_*.json', recursive=True):
    dst = os.path.join('data/kaggle', os.path.basename(f))
    if not os.path.exists(dst):
        os.symlink(f, dst)
print(os.listdir('data/kaggle'))

In [ ]:
# EPOP (тексты train/dev/test + разметка train/dev), GeoNames, NCBI Taxonomy -> data/
!python download_data.py

## 1. Проверка на dev (параметры графа и пороги подбираются только на train; в конце печатается метрика Kaggle)

In [ ]:
!python run.py dev

## 2. Подбор на train+dev и предсказание для test

In [ ]:
!python run.py test
!cp outputs/test/submission.csv /kaggle/working/submission.csv
!cd outputs && zip -qr /kaggle/working/pestclef_outputs.zip test
!head -c 600 /kaggle/working/submission.csv

## 3. Проверка сабмита

`/kaggle/working/submission.csv`: колонки `doc_id`, `knowledge_graph`. Во второй колонке JSON-массив рёбер
`{"predicate", "subject", "object"}` с одной строкой на узел. Его можно отправлять через Submit to Competition.

In [ ]:
import json
import pandas as pd

sub = pd.read_csv('/kaggle/working/submission.csv', dtype={'doc_id': str})
print(sub.shape, 'edges:', sub.knowledge_graph.map(lambda s: len(json.loads(s))).sum())
pd.DataFrame(json.loads(sub.knowledge_graph[0]))